In [8]:
!wget -nc https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/


--2024-04-08 09:52:50--  https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 400 Bad Request
2024-04-08 09:52:51 ERROR 400: Bad Request.



In [1]:
import numpy as np
from matplotlib import pyplot as plt
import string
from sklearn.model_selection import train_test_split

In [9]:
input_files = {
    '/content/edgar_allan_poe.txt',
    '/content/robert_frost.txt',
}

In [6]:
!head edgar_allen_poe.txt

head: cannot open 'edgar_allen_poe.txt' for reading: No such file or directory


In [10]:
input_files

{'/content/edgar_allan_poe.txt', '/content/robert_frost.txt'}

In [11]:
# Collect data into lists

input_texts = []
labels = []

for label, f in enumerate(input_files):
  print(f"{f} corresponds to label {label}")

  for line in open(f):
    line = line.rstrip().lower()
    if line:
      # remove punctuation

      line = line.translate(str.maketrans('','', string.punctuation))
      input_texts.append(line)
      labels.append(label)


/content/robert_frost.txt corresponds to label 0
/content/edgar_allan_poe.txt corresponds to label 1


In [12]:
train_text, test_text, Ytrain, Ytest = train_test_split(input_texts, labels
                                                        )

In [13]:
len(Ytrain), len(Ytest)

(1615, 539)

In [14]:
train_text[:5]


['her early leafs a flower',
 'in separate coops having their plumage done',
 'these were days when my heart was volcanic',
 'half looking for the orchid calypso',
 'and some exclaimed who saw afar']

In [15]:
Ytrain[:5]

[0, 0, 1, 0, 0]

In [16]:
idx = 1
word2idx = {'<unk': 0}

In [17]:
# populate word2idx

for text in train_text:
  tokens = text.split()
  for token in tokens:
    if token not in word2idx:
      word2idx[token] = idx
      idx+=1


In [18]:
word2idx

{'<unk': 0,
 'her': 1,
 'early': 2,
 'leafs': 3,
 'a': 4,
 'flower': 5,
 'in': 6,
 'separate': 7,
 'coops': 8,
 'having': 9,
 'their': 10,
 'plumage': 11,
 'done': 12,
 'these': 13,
 'were': 14,
 'days': 15,
 'when': 16,
 'my': 17,
 'heart': 18,
 'was': 19,
 'volcanic': 20,
 'half': 21,
 'looking': 22,
 'for': 23,
 'the': 24,
 'orchid': 25,
 'calypso': 26,
 'and': 27,
 'some': 28,
 'exclaimed': 29,
 'who': 30,
 'saw': 31,
 'afar': 32,
 'from': 33,
 'advantage': 34,
 'on': 35,
 'hill': 36,
 'to': 37,
 'athens': 38,
 'deliverance': 39,
 'gave': 40,
 'so': 41,
 'suddenly': 42,
 'i': 43,
 'flung': 44,
 'door': 45,
 'wide': 46,
 'him': 47,
 'along': 48,
 'that': 49,
 'wilderness': 50,
 'of': 51,
 'glass': 52,
 'make': 53,
 'it': 54,
 'worth': 55,
 'lifes': 56,
 'while': 57,
 'wake': 58,
 'sport': 59,
 'mountain': 60,
 'may': 61,
 'have': 62,
 'shifted': 63,
 'since': 64,
 'skies': 65,
 'they': 66,
 'ashen': 67,
 'sober': 68,
 'summer': 69,
 'passed': 70,
 'place': 71,
 'his': 72,
 'spirit':

In [19]:
len(word2idx)

2545

In [20]:
# Convert data into integer format

train_text_int = []
test_text_int = []

for text in train_text:
  tokens = text.split()
  line_as_int = {word2idx[token] for token in tokens}
  train_text_int.append(line_as_int)

for text in test_text:
  tokens= text.split()
  line_as_int = [word2idx.get(token , 0) for token in tokens]
  test_text_int.append(line_as_int)


In [21]:
train_text_int[100:105]

[{24, 51, 103, 137, 377, 378, 379, 380},
 {17, 23, 90, 318, 381, 382, 383},
 {14, 24, 49, 137, 384, 385, 386},
 {24, 118, 183, 193, 387, 388, 389, 390, 391},
 {37, 76, 90, 129, 154, 192, 202, 392, 393, 394}]

In [22]:
# Initialize A and pi matrices - for both classes

V = len(word2idx)
A0= np.ones((V,V))
pi0 = np.ones(V)

A1 = np.ones((V,V))
pi1 = np.ones(V)

In [25]:
# compute counts for A and pi

def compute_counts(text_as_int, A, pi):
  for tokens in text_as_int:
    last_idx = None
    for idx in tokens:
      if last_idx is None:
        # Its the first word in a sentence
        pi[idx]+=1
      else:
        # the last word exists, so count a transition
        A[last_idx, idx] += 1

      last_idx = idx


In [28]:
compute_counts([t for t, y in zip(train_text_int, Ytrain) if y == 0], A0, pi0)
compute_counts([t for t, y in zip(train_text_int, Ytrain) if y == 0], A1, pi1)

In [29]:
# Normalize A and pi o they are valid probability distribution

A0 /= A0.sum(axis =1, keepdims = True)
pi0 /= pi0.sum()

A1 /= A1.sum(axis =1, keepdims = True)
pi1 /= pi1.sum()


In [30]:
# log A and pi since we don't need the actual probs

logA0= np.log(A0)
logpi0 = np.log(pi0)

logA1 = np.log(A1)
logpi1 = np.log(pi1)

In [32]:
# Compute priors

count0 = sum(y == 0 for y in Ytrain)
count1 = sum(y ==1 for y in Ytrain)

total = len(Ytrain)
p0 = count0/ total
p1 = count1 / total
logp0 = np.log(p0)
logp1 = np.log(p1)

p0, p1

(0.6643962848297214, 0.33560371517027865)

In [53]:
class Classifier:
    def __init__(self, logAs, logpis, logpriors):
        self.logAs = logAs
        self.logpis = logpis
        self.logpriors = logpriors
        self.K = len(logpriors)

    def _compute_log_likelihood(self, input_, class_):
        logA = self.logAs[class_]  # Access list element directly
        logpi = self.logpis[class_]  # Access list element directly

        last_idx = None
        logprob = 0
        for idx in input_:
            if last_idx is None:
                logprob += logpi[idx]
            else:
                logprob += logA[last_idx,idx]  # Accessing logA as 2D list

            last_idx = idx

        return logprob

    def predict(self, inputs):
        predictions = np.zeros(len(inputs))
        for i, input_ in enumerate(inputs):
            posteriors = [self._compute_log_likelihood(input_, c) + self.logpriors[c]
                          for c in range(self.K)]
            pred = np.argmax(posteriors)  # Use posteriors to determine prediction
            predictions[i] = pred
        return predictions


In [54]:
# each array must be in order since classes are assumed to index these lists

clf = Classifier([logA0, logA1], [logpi0, logpi1], [logp0, logp1])

In [55]:
Ptrain = clf.predict(train_text_int)
print(f"Train acc: {np.mean(Ptrain == Ytrain)}")

Train acc: 0.6643962848297214


In [56]:
Ptest = clf.predict(test_text_int)
print(f"Test acc: {np.mean(Ptest == Ytest)}")

Test acc: 0.673469387755102


In [57]:
from sklearn.metrics import confusion_matrix, f1_score

In [58]:
cm = confusion_matrix(Ytrain, Ptrain)
cm

array([[1073,    0],
       [ 542,    0]])

In [59]:
cm_test = confusion_matrix(Ytest, Ptest)
cm_test

array([[363,   0],
       [176,   0]])

In [60]:
f1_score(Ytrain , Ptrain)

0.0

In [61]:
f1_score(Ytest, Ptest)

0.0